# 06 Visualization Upgrade

영화 흥행 데이터의 분포, 성공도 랭크, 모델 오분류 구조를 보고서에 바로 넣을 수 있는 시각화로 생성합니다.

생성되는 주요 그림:

- 영화 feature PCA 2D 분포맵
- 실제 관객 수 vs 예측 성공도 랭크 산점도
- 성공도 랭크 confusion heatmap
- 장르 x 성공도 랭크 heatmap
- 개봉 월 x 성공도 랭크 heatmap
- 포스터 밝기/채도 스타일 산점도
- 오분류 영화 lollipop chart


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from preprocessing import experiment_feature_columns

DATA_PATH = ROOT / 'data/processed/movie_features.csv'
RANK_METRICS_PATH = ROOT / 'outputs/metrics/success_rank_classification_metrics.csv'
RANK_PREDICTIONS_PATH = ROOT / 'outputs/metrics/success_rank_classification_predictions.csv'
FIGURE_DIR = ROOT / 'outputs/figures/visualization_upgrade'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANK_ORDER = ['D', 'C', 'B', 'A', 'S']
RANK_CODE = {rank: index for index, rank in enumerate(RANK_ORDER)}
RANK_PALETTE = {
    'D': '#8d99ae',
    'C': '#4c78a8',
    'B': '#59a14f',
    'A': '#f28e2b',
    'S': '#d62728',
}
STATE_PALETTE = {
    'correct': '#2f7d32',
    'incorrect': '#c62828',
    'true_positive': '#2f7d32',
    'true_negative': '#4c78a8',
    'false_positive': '#ef8a24',
    'false_negative': '#c62828',
}

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.unicode_minus'] = False
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'DejaVu Sans']:
    plt.rcParams['font.family'] = font_name
    break

def save_current(name: str):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print(path.relative_to(ROOT))
    return path

print(ROOT)
print(FIGURE_DIR.relative_to(ROOT))

## 1. 데이터 준비

노트북을 단독으로 실행해도 필요한 전처리/예측 파일이 없으면 먼저 생성합니다.

In [ ]:
def run_if_missing(path: Path, command: list[str]):
    if path.exists():
        return
    print(f'생성 중: {path.relative_to(ROOT)}')
    subprocess.run(command, cwd=ROOT, check=True)

run_if_missing(
    DATA_PATH,
    [sys.executable, 'src/preprocessing.py', '--input', 'data/kobis_tmdb_title_matches.csv', '--output', str(DATA_PATH)],
)
run_if_missing(
    RANK_PREDICTIONS_PATH,
    [
        sys.executable,
        'src/train_classification.py',
        '--input', str(DATA_PATH),
        '--target', 'success_rank',
        '--output', str(RANK_METRICS_PATH),
        '--figures-dir', 'outputs/figures/success_rank_confusion',
        '--predictions-output', str(RANK_PREDICTIONS_PATH),
        '--prediction-figures-dir', 'outputs/figures/success_rank_predictions',
    ],
)

movies = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
rank_metrics = pd.read_csv(RANK_METRICS_PATH, encoding='utf-8-sig')
rank_predictions = pd.read_csv(RANK_PREDICTIONS_PATH, encoding='utf-8-sig')

movies['success_rank'] = pd.Categorical(movies['success_rank'], categories=RANK_ORDER, ordered=True)
movies['audience_million'] = movies['audience_count'] / 1_000_000
movies['log_audience'] = np.log10(movies['audience_count'].clip(lower=1))

best_rank_model = (
    rank_metrics.sort_values(['f1_macro', 'accuracy'], ascending=False)
    .iloc[0][['experiment', 'model', 'f1_macro', 'accuracy']]
)
best_predictions = rank_predictions[
    (rank_predictions['experiment'] == best_rank_model['experiment'])
    & (rank_predictions['model'] == best_rank_model['model'])
].copy()
best_predictions['actual_code'] = best_predictions['actual_label'].map(RANK_CODE)
best_predictions['predicted_code'] = best_predictions['predicted_label'].map(RANK_CODE)
best_predictions['rank_gap'] = (best_predictions['predicted_code'] - best_predictions['actual_code']).abs()
best_predictions['log_audience'] = np.log10(best_predictions['audience_count'].clip(lower=1))

print('movies:', movies.shape)
print('rank counts')
print(movies['success_rank'].value_counts().reindex(RANK_ORDER).fillna(0).astype(int))
print('\nbest rank model')
print(best_rank_model.to_string())

## 2. 성공도 랭크 분포

전체 데이터가 얼마나 불균형한지 보여주는 기본 그림입니다. 모델 성능 해석 전에 넣기 좋습니다.

In [ ]:
rank_counts = movies['success_rank'].value_counts().reindex(RANK_ORDER).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(rank_counts.index, rank_counts.values, color=[RANK_PALETTE[r] for r in rank_counts.index], edgecolor='#333333', linewidth=0.7)
for bar, value in zip(bars, rank_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 4, f'{value}', ha='center', va='bottom', fontsize=10)
ax.set_title('성공도 랭크 분포')
ax.set_subtitle = None
ax.set_xlabel('성공도 랭크')
ax.set_ylabel('영화 수')
ax.grid(axis='y', alpha=0.25)
save_current('rank_distribution.png')

## 3. 영화 Feature PCA 2D 분포맵

메타데이터, 장르, 제목, 포스터 특성을 모두 사용해 영화를 2차원으로 압축합니다. 점 하나가 영화 1편이며, 색은 성공도 랭크, 크기는 관객 수입니다.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_columns = experiment_feature_columns(movies, 'all')
feature_matrix = movies[feature_columns].apply(pd.to_numeric, errors='coerce')
pca_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), PCA(n_components=2, random_state=42))
pca_values = pca_pipe.fit_transform(feature_matrix)
explained = pca_pipe.named_steps['pca'].explained_variance_ratio_
plot_frame = movies[['match_title', 'success_rank', 'audience_count', 'audience_million']].copy()
plot_frame['pca_1'] = pca_values[:, 0]
plot_frame['pca_2'] = pca_values[:, 1]
plot_frame['point_size'] = 22 + np.log10(plot_frame['audience_count'].clip(lower=1)) * 10

fig, ax = plt.subplots(figsize=(10, 7))
for rank in RANK_ORDER:
    group = plot_frame[plot_frame['success_rank'] == rank]
    ax.scatter(
        group['pca_1'],
        group['pca_2'],
        s=group['point_size'],
        color=RANK_PALETTE[rank],
        alpha=0.72,
        edgecolor='white',
        linewidth=0.4,
        label=f'{rank} ({len(group)})',
    )

for _, row in plot_frame.nlargest(12, 'audience_count').iterrows():
    ax.text(row['pca_1'], row['pca_2'], row['match_title'], fontsize=8, alpha=0.9)

ax.set_title('영화 특성 기반 2D 분포맵')
ax.set_xlabel(f'PCA 1 ({explained[0]:.1%})')
ax.set_ylabel(f'PCA 2 ({explained[1]:.1%})')
ax.grid(alpha=0.22)
ax.legend(title='성공도 랭크', loc='best', fontsize=9)
save_current('movie_feature_pca_map.png')

## 4. 실제 관객 수 vs 예측 성공도 랭크

관객 수가 큰데 낮게 예측한 영화, 관객 수가 작은데 높게 예측한 영화를 빠르게 찾는 그림입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for result_name, group in best_predictions.groupby('classification_result'):
    ax.scatter(
        group['audience_count'],
        group['predicted_code'],
        color=STATE_PALETTE.get(result_name, '#777777'),
        alpha=0.78,
        s=54,
        edgecolor='white',
        linewidth=0.5,
        label=f'{result_name} ({len(group)})',
    )

for _, row in best_predictions[best_predictions['classification_result'] == 'incorrect'].nlargest(10, 'audience_count').iterrows():
    ax.text(row['audience_count'], row['predicted_code'] + 0.06, row['match_title'], fontsize=8)

for threshold in [1_000_000, 3_000_000, 5_000_000, 10_000_000]:
    ax.axvline(threshold, color='#666666', linestyle='--', linewidth=0.8, alpha=0.55)
ax.set_xscale('log')
ax.set_yticks(range(len(RANK_ORDER)))
ax.set_yticklabels(RANK_ORDER)
ax.set_xlabel('실제 관객 수 (log scale)')
ax.set_ylabel('예측 성공도 랭크')
ax.set_title(f'실제 관객 수 vs 예측 랭크 ({best_rank_model["experiment"]} / {best_rank_model["model"]})')
ax.grid(True, axis='x', which='both', alpha=0.18)
ax.grid(True, axis='y', alpha=0.2)
ax.legend(loc='best', fontsize=9)
save_current('actual_audience_vs_predicted_rank.png')

## 5. 성공도 랭크 Confusion Heatmap

행 기준 비율로 정규화해 실제 랭크가 어느 랭크로 헷갈리는지 봅니다.

In [ ]:
confusion_counts = pd.crosstab(
    best_predictions['actual_label'],
    best_predictions['predicted_label'],
).reindex(index=RANK_ORDER, columns=RANK_ORDER, fill_value=0)
confusion_rate = confusion_counts.div(confusion_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(confusion_rate.values, cmap='Blues', vmin=0, vmax=max(0.01, confusion_rate.values.max()))
ax.set_xticks(range(len(RANK_ORDER)))
ax.set_xticklabels(RANK_ORDER)
ax.set_yticks(range(len(RANK_ORDER)))
ax.set_yticklabels(RANK_ORDER)
ax.set_xlabel('예측 랭크')
ax.set_ylabel('실제 랭크')
ax.set_title('성공도 랭크 오분류 구조')
for i in range(len(RANK_ORDER)):
    for j in range(len(RANK_ORDER)):
        count = int(confusion_counts.iloc[i, j])
        pct = confusion_rate.iloc[i, j]
        color = 'white' if pct > confusion_rate.values.max() * 0.55 else '#222222'
        ax.text(j, i, f'{count}\n{pct:.0%}', ha='center', va='center', fontsize=9, color=color)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='행 기준 비율')
save_current('rank_confusion_heatmap.png')

## 6. 장르 x 성공도 랭크 Heatmap

장르별로 상위 랭크 비중이 어떻게 다른지 봅니다. 표본이 너무 작은 장르는 제외합니다.

In [ ]:
genre_rows = []
for _, row in movies.iterrows():
    genres = [genre.strip() for genre in str(row.get('tmdb_genres', '')).split('|') if genre.strip()]
    for genre in genres:
        genre_rows.append({'genre': genre, 'success_rank': row['success_rank'], 'audience_count': row['audience_count']})

genre_frame = pd.DataFrame(genre_rows)
genre_counts = genre_frame['genre'].value_counts()
selected_genres = genre_counts[genre_counts >= 20].index.tolist()
genre_rank = pd.crosstab(
    genre_frame[genre_frame['genre'].isin(selected_genres)]['genre'],
    genre_frame[genre_frame['genre'].isin(selected_genres)]['success_rank'],
).reindex(columns=RANK_ORDER, fill_value=0)
genre_rank_share = genre_rank.div(genre_rank.sum(axis=1), axis=0).sort_values(['S', 'A', 'B'], ascending=False)

fig, ax = plt.subplots(figsize=(8, max(5, len(genre_rank_share) * 0.38)))
im = ax.imshow(genre_rank_share.values, cmap='YlGnBu', aspect='auto', vmin=0, vmax=genre_rank_share.values.max())
ax.set_xticks(range(len(RANK_ORDER)))
ax.set_xticklabels(RANK_ORDER)
ax.set_yticks(range(len(genre_rank_share.index)))
ax.set_yticklabels(genre_rank_share.index)
ax.set_xlabel('성공도 랭크')
ax.set_title('장르별 성공도 랭크 구성비')
for i in range(len(genre_rank_share.index)):
    for j in range(len(RANK_ORDER)):
        value = genre_rank_share.iloc[i, j]
        if value >= 0.12:
            ax.text(j, i, f'{value:.0%}', ha='center', va='center', fontsize=8, color='#111111')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='장르 내 비중')
save_current('genre_rank_heatmap.png')

## 7. 개봉 월 x 성공도 랭크 Heatmap

성수기 효과를 보기 위한 월별 랭크 구성비입니다.

In [ ]:
month_rank = pd.crosstab(movies['meta_open_month'], movies['success_rank']).reindex(index=range(1, 13), columns=RANK_ORDER, fill_value=0)
month_rank_share = month_rank.div(month_rank.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(month_rank_share.values, cmap='Oranges', aspect='auto', vmin=0, vmax=month_rank_share.values.max())
ax.set_xticks(range(len(RANK_ORDER)))
ax.set_xticklabels(RANK_ORDER)
ax.set_yticks(range(12))
ax.set_yticklabels([f'{m}월' for m in range(1, 13)])
ax.set_xlabel('성공도 랭크')
ax.set_ylabel('개봉 월')
ax.set_title('개봉 월별 성공도 랭크 구성비')
for i in range(12):
    for j in range(len(RANK_ORDER)):
        value = month_rank_share.iloc[i, j]
        if value >= 0.15:
            ax.text(j, i, f'{value:.0%}', ha='center', va='center', fontsize=8, color='#111111')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='월 내 비중')
save_current('month_rank_heatmap.png')

## 8. 포스터 밝기/채도 스타일 산점도

포스터의 저수준 시각 특성과 흥행 랭크가 분리되는지 탐색합니다. 강한 인과 해석보다는 보조 분석용입니다.

In [ ]:
poster_frame = movies.dropna(subset=['poster_brightness_mean', 'poster_saturation_mean']).copy()
poster_frame['point_size'] = 26 + poster_frame['log_audience'] * 9
fig, ax = plt.subplots(figsize=(9, 6.5))
for rank in RANK_ORDER:
    group = poster_frame[poster_frame['success_rank'] == rank]
    ax.scatter(
        group['poster_brightness_mean'],
        group['poster_saturation_mean'],
        s=group['point_size'],
        color=RANK_PALETTE[rank],
        alpha=0.72,
        edgecolor='white',
        linewidth=0.4,
        label=f'{rank} ({len(group)})',
    )
for _, row in poster_frame.nlargest(10, 'audience_count').iterrows():
    ax.text(row['poster_brightness_mean'], row['poster_saturation_mean'], row['match_title'], fontsize=8)
ax.set_xlabel('포스터 평균 밝기')
ax.set_ylabel('포스터 평균 채도')
ax.set_title('포스터 시각 특성과 성공도 랭크')
ax.grid(alpha=0.22)
ax.legend(title='성공도 랭크', loc='best', fontsize=9)
save_current('poster_brightness_saturation_scatter.png')

## 9. 오분류 영화 Lollipop

관객 수가 큰 오분류 사례를 우선 보여줍니다. 보고서의 한계/개선 방향 섹션에 넣기 좋습니다.

In [ ]:
misses = best_predictions[best_predictions['classification_result'] == 'incorrect'].copy()
misses = misses.sort_values(['audience_count', 'rank_gap'], ascending=False).head(14).sort_values('audience_count')

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(misses))
for i, (_, row) in enumerate(misses.iterrows()):
    ax.plot([row['actual_code'], row['predicted_code']], [i, i], color='#999999', linewidth=1.8, zorder=1)
    ax.scatter(row['actual_code'], i, color=RANK_PALETTE[row['actual_label']], s=80, edgecolor='white', linewidth=0.5, zorder=2, label='actual' if i == 0 else None)
    ax.scatter(row['predicted_code'], i, color='white', s=90, edgecolor=RANK_PALETTE[row['predicted_label']], linewidth=2, zorder=3, label='predicted' if i == 0 else None)

labels = [f"{row.match_title} ({row.audience_count/1_000_000:.1f}M)" for row in misses.itertuples()]
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xticks(range(len(RANK_ORDER)))
ax.set_xticklabels(RANK_ORDER)
ax.set_xlabel('성공도 랭크')
ax.set_title('관객 수가 큰 오분류 영화')
ax.grid(axis='x', alpha=0.25)
ax.legend(loc='lower right', fontsize=9)
save_current('misclassified_movies_lollipop.png')

## 10. 저장된 파일 목록

In [ ]:
created_files = sorted(FIGURE_DIR.glob('*.png'))
for path in created_files:
    print(path.relative_to(ROOT))